# Multilingual Health QA — Fine-Tuning mT5-base
## Experiment 2: Language-Aware Seq2Seq Fine-Tuning

**Author:** Samuel Mwania
**Competition:** Multilingual Health QA in Low-Resource African Languages — Zindi
**Environment:** Kaggle T4 GPU

### Hypothesis
A properly fine-tuned mT5-base model with language-aware prompting will learn to generate
fluent, contextually appropriate health answers in the correct language, beating the
TF-IDF retrieval baseline (0.4908).

### Key design decisions (justified by EDA in Notebook 01)
- **Model:** google/mt5-base — pretrained on 101 languages including all 5 in this dataset
- **Input max length:** 128 tokens (covers 99th percentile of question lengths)
- **Target max length:** 256 tokens (covers ~95th percentile of answer lengths)
- **Language-aware prompt:** prepend language name so model knows target language
  (fixes the disabled-prompt bug in the starter notebook)
- **Precision:** bf16 (T4 supports it, stable training)

In [2]:
# Force single-GPU usage to avoid inefficient DataParallel memory overflow
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("Restricted to GPU 0 only")

Restricted to GPU 0 only


In [3]:
import torch
print("Visible GPUs:", torch.cuda.device_count())
free, total = torch.cuda.mem_get_info()
print(f"Free GPU memory: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

Visible GPUs: 1
Free GPU memory: 15.53 GB / 15.64 GB


In [4]:
# Install required versions
!pip install -q -U transformers==4.44.2 datasets==2.21.0 sentencepiece==0.2.0 accelerate==0.34.2
print("Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 61.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 74.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigfra

In [5]:
# Imports and environment check
import os
import re
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Check GPU
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("bf16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

# Set seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("\nSeed set to", SEED)

2026-06-05 00:18:01.427595: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780618681.888515      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780618681.999470      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780618683.127098      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780618683.127146      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780618683.127149      58 computation_placer.cc:177] computation placer alr

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
bf16 supported: True

Seed set to 42


In [6]:
# Locate the input data files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/samuelmwania1/multilingual-health-qa-data/Val.csv
/kaggle/input/datasets/samuelmwania1/multilingual-health-qa-data/SampleSubmission.csv
/kaggle/input/datasets/samuelmwania1/multilingual-health-qa-data/Train.csv
/kaggle/input/datasets/samuelmwania1/multilingual-health-qa-data/Test.csv


## 1. Load and Clean Data

Load the competition data and apply the same conservative cleaning strategy
documented in Notebook 01 (EDA): drop the single empty-input record and exact
duplicate pairs, while preserving valid short factual answers. This keeps the
training pipeline consistent with my exploratory analysis.

In [7]:
# Load data
DATA_DIR = '/kaggle/input/datasets/samuelmwania1/multilingual-health-qa-data'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'Val.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'Test.csv'))

print("Raw shapes:")
print(f"  Train: {train_df.shape}")
print(f"  Val:   {val_df.shape}")
print(f"  Test:  {test_df.shape}")

# Conservative cleaning (matches Notebook 01)
def clean_training_data(df):
    before = len(df)
    # Drop empty inputs
    df = df[df['input'].str.split().str.len() > 0].copy()
    # Drop exact duplicate pairs, keep first
    df = df.drop_duplicates(subset=['input', 'output'], keep='first').reset_index(drop=True)
    print(f"  Cleaned: {before} -> {len(df)} (removed {before - len(df)})")
    return df

print("\nCleaning train:")
train_df = clean_training_data(train_df)
print("Cleaning val:")
val_df = clean_training_data(val_df)

print(f"\nFinal: {len(train_df):,} train | {len(val_df):,} val | {len(test_df):,} test")
print(f"\nLanguages: {sorted(train_df['subset'].unique())}")

Raw shapes:
  Train: (29815, 4)
  Val:   (6686, 4)
  Test:  (2618, 3)

Cleaning train:
  Cleaned: 29815 -> 29538 (removed 277)
Cleaning val:
  Cleaned: 6686 -> 6673 (removed 13)

Final: 29,538 train | 6,673 val | 2,618 test

Languages: ['Aka_Gha', 'Amh_Eth', 'Eng_Eth', 'Eng_Gha', 'Eng_Ken', 'Eng_Uga', 'Lug_Uga', 'Swa_Ken']


## 2. Language-Aware Prompt Construction

The starter notebook disabled language conditioning (its `build_prompt` returned the
raw question). I will restore and improve it: each input is prefixed with a human-readable
language tag derived from the `subset` code. This explicitly signals the target
language and country context to the model, which is critical for a multilingual
seq2seq model that must answer in the same language as the question.

Example: `Aka_Gha` question becomes:
`"Answer this health question in Akan (Ghana): <question>"`

In [8]:
# Map subset codes to readable language + country names
SUBSET_TO_LANG = {
    'Aka_Gha': 'Akan (Ghana)',
    'Amh_Eth': 'Amharic (Ethiopia)',
    'Eng_Eth': 'English (Ethiopia)',
    'Eng_Gha': 'English (Ghana)',
    'Eng_Ken': 'English (Kenya)',
    'Eng_Uga': 'English (Uganda)',
    'Lug_Uga': 'Luganda (Uganda)',
    'Swa_Ken': 'Swahili (Kenya)',
}

def build_prompt(question, subset):
    """Construct a language-aware prompt for the model."""
    lang = SUBSET_TO_LANG.get(subset, 'the same language')
    return f"Answer this health question in {lang}: {str(question).strip()}"

# Apply to all splits
train_df['prompt'] = train_df.apply(lambda r: build_prompt(r['input'], r['subset']), axis=1)
val_df['prompt']   = val_df.apply(lambda r: build_prompt(r['input'], r['subset']), axis=1)
test_df['prompt']  = test_df.apply(lambda r: build_prompt(r['input'], r['subset']), axis=1)

# Show examples across languages
print("PROMPT EXAMPLES")
print("=" * 80)
for subset in ['Aka_Gha', 'Amh_Eth', 'Swa_Ken', 'Eng_Uga']:
    example = train_df[train_df['subset'] == subset].iloc[0]
    print(f"[{subset}]")
    print(f"Prompt: {example['prompt'][:120]}")
    print(f"Target: {example['output'][:100]}")
    print("-" * 80)

PROMPT EXAMPLES
[Aka_Gha]
Prompt: Answer this health question in Akan (Ghana): Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ w
Target: Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa 
--------------------------------------------------------------------------------
[Amh_Eth]
Prompt: Answer this health question in Amharic (Ethiopia): የረጅም ጊዜ ጤናን በተመለከተ በጣም ሊያሳስበኝ የሚገቡ ሶስት ዋና ዋና ኢንፌክሽኖች የትኞቹ ናቸው?
Target: ኤች አይ ቪ፣ ሄፓታይተስ ቢ እና ሂውማን ፓፒሎማ ቫይረስ (HPV)ናቸው።
--------------------------------------------------------------------------------
[Swa_Ken]
Prompt: Answer this health question in Swahili (Kenya): Je, PrEP inatoa ulinzi kweli, au imani katika ufanisi wake inachukuliwa 
Target: PrEP (Pre-Exposure Prophylaxis) sio hadithi; ni mkakati wa kweli na madhubuti wa kuzuia virusi vya u
--------------------------------------------------------------------------------
[Eng_Uga]
Prompt: Answer this health question in English (U

## 3. Load Model and Tokenizer

Load google/mt5-base (580M parameters). mT5 was pretrained on the mC4 corpus covering
101 languages, including all five languages in this competition. This multilingual
pretraining is why mT5 is well-suited to the task versus an English-only model.

In [9]:
MODEL_NAME = 'google/mt5-base'

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params/1e6:.1f}M parameters")
print(f"Device: {device}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

Loading google/mt5-base...


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: 582.4M parameters
Device: cuda
Vocab size: 250,100


## 4. Tokenization

Tokenize using the length limits derived from EDA:
- **Input (max 128 tokens):** covers the 99th percentile of question lengths
- **Target (max 256 tokens):** covers roughly the 95th percentile of answer lengths

Truncation beyond these limits affects under 5% of examples while keeping memory
usage efficient. I use the modern `text_target` API for label tokenization and mask
padding tokens to -100 so they are ignored by the loss function.

In [10]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    # Tokenize inputs
    model_inputs = tokenizer(
        examples['prompt'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False  # dynamic padding handled by collator
    )
    # Tokenize targets using the modern API
    labels = tokenizer(
        text_target=examples['output'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df[['prompt', 'output']])
val_dataset   = Dataset.from_pandas(val_df[['prompt', 'output']])

# Tokenize
print("Tokenizing train...")
train_tok = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
print("Tokenizing val...")
val_tok = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

print(f"\nTokenized train: {len(train_tok)} examples")
print(f"Tokenized val:   {len(val_tok)} examples")
print(f"\nSample tokenized input length: {len(train_tok[0]['input_ids'])}")
print(f"Sample tokenized label length: {len(train_tok[0]['labels'])}")

Tokenizing train...


Map:   0%|          | 0/29538 [00:00<?, ? examples/s]

Tokenizing val...


Map:   0%|          | 0/6673 [00:00<?, ? examples/s]


Tokenized train: 29538 examples
Tokenized val:   6673 examples

Sample tokenized input length: 128
Sample tokenized label length: 256


## 5. Training Configuration

Hyperparameters and their justification:
- **Epochs: 3** -> enough to adapt mT5 to the domain without overfitting on 29.5K examples
- **Batch size: 8** with **gradient accumulation: 2** -> effective batch size 16, fits T4 memory
- **Learning rate: 5e-5** -> standard for mT5 fine-tuning, stable convergence
- **bf16: True** -> T4 supports it, halves memory with stable numerics
- **Warmup: 500 steps** -> gradual LR increase prevents early instability
- **Eval/save each epoch**, keep best model on lowest eval_loss
- **Validation subset** -> a stratified 1,000-example sample for fast per-epoch evaluation

In [11]:
# Create a stratified validation subset for faster evaluation during training
val_subset_df = (
    val_df.groupby('subset', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 125), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Validation subset: {len(val_subset_df)} examples")
print(val_subset_df['subset'].value_counts().sort_index())

val_subset_dataset = Dataset.from_pandas(val_subset_df[['prompt', 'output']])
val_subset_tok = val_subset_dataset.map(
    preprocess_function, batched=True, remove_columns=val_subset_dataset.column_names
)
print(f"\nTokenized validation subset: {len(val_subset_tok)}")

Validation subset: 1000 examples
subset
Aka_Gha    125
Amh_Eth    125
Eng_Eth    125
Eng_Gha    125
Eng_Ken    125
Eng_Uga    125
Lug_Uga    125
Swa_Ken    125
Name: count, dtype: int64


/tmp/ipykernel_58/3041199136.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 125), random_state=SEED))


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Tokenized validation subset: 1000


In [12]:
# Data collator handles dynamic padding and label masking
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

# Training arguments
OUTPUT_DIR = '/kaggle/working/mt5_base_exp2'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='adafactor',
    save_safetensors=False,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=False,
    report_to='none',
    seed=SEED
)

print("Training configuration set")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Output dir: {OUTPUT_DIR}")

Training configuration set
Effective batch size: 16
Output dir: /kaggle/working/mt5_base_exp2


## 6. Training

Initialize the Seq2SeqTrainer and fine-tune for 3 epochs. The best model (lowest
validation loss) is automatically loaded at the end. Training loss and validation
loss are logged to track convergence and detect overfitting.

In [13]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_subset_tok,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Starting training...")
print(f"Total training examples: {len(train_tok):,}")
print(f"Estimated steps: {len(train_tok) // 16 * 3:,}")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Final train loss: {train_result.training_loss:.4f}")

Starting training...
Total training examples: 29,538
Estimated steps: 5,538


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
0,2.960700,2.373687
1,2.592400,2.165679
2,2.500800,2.116652



TRAINING COMPLETE
Final train loss: 3.4568


## 7. Inference and Submission Generation

With the fine-tuned model loaded (best checkpoint by validation loss), generate
answers for the 2,618 test questions, using the same language-aware prompts as training,
and beam search decoding for fluent, higher-quality generation.

### Generation settings (and why)
- **num_beams=4** — beam search explores multiple candidate sequences, improving fluency and ROUGE over greedy decoding
- **max_new_tokens=256** — matches training target length; allows full answers for long-answer languages (Eng_Uga, Akan)
- **no_repeat_ngram_size=3** — prevents degenerate repetition loops common in fine-tuned seq2seq models
- **length_penalty=1.0** — neutral starting point; tuned in a later experiment

In [14]:
# Ensure model is in eval mode and cache re-enabled for fast generation
model.config.use_cache = True
model.eval()

from torch.utils.data import DataLoader

def generate_answers_batch(prompts, batch_size=16, num_beams=4, max_new_tokens=256):
    """Generate answers for a list of prompts using batched beam search."""
    all_answers = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=num_beams,
                max_new_tokens=max_new_tokens,
                no_repeat_ngram_size=3,
                length_penalty=1.0,
                early_stopping=True
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_answers.extend(decoded)

        if (i // batch_size) % 10 == 0:
            print(f"  Generated {min(i+batch_size, len(prompts))}/{len(prompts)}")
    return all_answers

print("Generating answers for test set...")
test_prompts = test_df['prompt'].tolist()
test_answers = generate_answers_batch(test_prompts)
print(f"\nDone. Generated {len(test_answers)} answers.")

Generating answers for test set...
  Generated 16/2618
  Generated 176/2618
  Generated 336/2618
  Generated 496/2618
  Generated 656/2618
  Generated 816/2618
  Generated 976/2618
  Generated 1136/2618
  Generated 1296/2618
  Generated 1456/2618
  Generated 1616/2618
  Generated 1776/2618
  Generated 1936/2618
  Generated 2096/2618
  Generated 2256/2618
  Generated 2416/2618
  Generated 2576/2618

Done. Generated 2618 answers.


In [16]:
# Clean generated text and inspect a few outputs across languages
def clean_generated(text):
    text = str(text).strip()
    # Remove leftover mT5 sentinel tokens
    text = re.sub(r'<extra_id_\d+>', '', text)
    # Remove the learned scaffolding prefix "This is a question about, X."
    text = re.sub(r'^This is a question about,?\s*[^.]*\.\s*', '', text, flags=re.IGNORECASE)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "No answer available"

test_answers_clean = [clean_generated(a) for a in test_answers]

# Inspect across languages
print("GENERATED ANSWER SAMPLES")
print("=" * 80)
for subset in ['Aka_Gha', 'Amh_Eth', 'Swa_Ken', 'Eng_Uga']:
    idx = test_df[test_df['subset'] == subset].index[0]
    pos = test_df.index.get_loc(idx)
    print(f"[{subset}]")
    print(f"Q: {test_df.loc[idx, 'input'][:90]}")
    print(f"A: {test_answers_clean[pos][:150]}")
    print("-" * 80)

GENERATED ANSWER SAMPLES
[Aka_Gha]
Q: Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwu
A: Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow de asiw GBV ano ma wɔbɛhwehwɛ nsɛm a ɛfa nna ho nkɔmmɔbɔ a wobetumi ayɛ nnwu
--------------------------------------------------------------------------------
[Amh_Eth]
Q: ክላሚዲያ ሳይታከም ቢቆይ በወንዶች ላይ የሚያስከትለው የረጅም ጊዜ ጉዳት ምንድን ነው?
A: የረጅም ጊዜ ጉዳት በወንዶች ላይ የሚከሰት የረጃም ግንኙነት ምንም ምልክት አይኖርም።
--------------------------------------------------------------------------------
[Swa_Ken]
Q: Je, mtu mwenye afya nzuri anaweza kuambukizwa virusi vya ukimwi/UKIMWI?
A: Ni muhimu kutambua kwamba mtu mwenye afya nzuri anaweza kuambukizwa virusi vya ukimwi (Virusi vya Upungufu wa Kinga ya Binadamu).
--------------------------------------------------------------------------------
[Eng_Uga]
Q: Treatment for Gonorrhea?, please answer this using simple medical terms.
A: No. However, there is no cure for gonorrhe

In [17]:
# Build the Zindi submission (same validated format as Experiment 1)
def make_submission(test_df, predictions, filepath):
    sub = pd.DataFrame({
        'ID': test_df['ID'].values,
        'TargetRLF1': predictions,
        'TargetR1F1': predictions,
        'TargetLLM': predictions
    })
    assert list(sub.columns) == ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert len(sub) == len(test_df)
    assert sub['ID'].equals(test_df['ID'])
    for col in ['TargetRLF1', 'TargetR1F1', 'TargetLLM']:
        sub[col] = sub[col].replace('', 'No answer available').fillna('No answer available')
    assert sub.isnull().sum().sum() == 0
    sub.to_csv(filepath, index=False, encoding='utf-8')
    print(f"Saved: {filepath} | {len(sub)} rows | validations passed")
    return sub

sub_exp2 = make_submission(test_df, test_answers_clean, '/kaggle/working/exp02_mt5base_finetuned.csv')

Saved: /kaggle/working/exp02_mt5base_finetuned.csv | 2618 rows | validations passed
